## VLM Shellgame multiview

- We want multiview videos such that:
  - videos test **correspondence** (unique object identification) AND **geometric understanding**   
  (cam position understanding) on the shell game.
  - One view should not be enough to predict.
  - Different cam positions not just two cams across from each other
  - single-view shellgame vids were 9-10s long. These multiview ones are slightly longer at 11-12s


- VLM is **gemini-3.8-flash (processing mode: 'agentic')** (model performed very well in single-view shellgame)  
→ A,B,C labeled on table, visible in cam1 (POV). At start, show where puck is, move camera to other view cam1,   
shuffle cups, move back into cam0 view and place on A B and C markers.
- 3 types of videos:
  - shellgame_mv_a – cam0 and cam1 look in same direction
  - shellgame_mv_b – cam0 and cam1 across from eachother (lateral inversion)
  - shellgame_mv_c – cam0 same and cam1 on some diagonal angle or even left or right view camera.
- For each video model picks from A,B,C


### Workflow for multiview eval
Compress cam0 and cam1 down to 480p-20fps --> Stitch both views together --> Write custom prompt --> API call

## Result:
- Not all videos are processing. 41 total videos and only (a+b+c = 16+8+3 = 27) vids processed. Rest threw timeout error even after more than 5 attempts per video.
- **top1acc shellgame_mv_a (0.75) > shellgame_mv_b (0.38) > shellgame_mv_c (0.0)** -- as expected
- CONCLUSION: Stitching together information from the 2 perspectives is something the model CANNOT do as evident from very poor performance on shellgame_mv_b and shellgame_mv_c.
- Close error analysis shellgame_mv_b showed 2 types of errors: 
  - A. In some videos model is follwoing the puck for a few moves correctly and then just abruptly returning result when more swaps are still happening (e.g. 261125-112001). 
  - B. Proper mistakes on model's part in tracking swaps (e.g. 261125-112139).


- no pattern in prompt/completion tokens.
- No real pattern in nframes seen by model. avg 100-120 frames_seen across vids. Highest is 205 frames (shellgame_mv_a: 261125-104759) still wrong prediction. swap sequence suggests the model just did not see all swaps because it was correct for first few swaps then just returned answer.
- **camera_setup_reasoning:** I asked in the prompt for the model to comment on what it thinks the camera setup is like or how it thinks cam0 is positioned relative to cam1. (I disabled it for quicker evals after 18 videos because it was enough)
  - On 12/13 shellgame_mv_a videos the reasoning is good. This was simplest setup.
  - On 2/3 shellgame_mv_b videos the reasoning is good. 
  - On 3/3 shellgame_mv_c videos reasoning is good.
  - CONCLUSION: I think geometric understanding of model is fairly decent in 2-camera setup insofar as knowing how the cameras are positioned. 




In [1]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

# import cv2

In [2]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with vqa videos

In [3]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('shellgame_mv')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam0'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(285, 4)
(41, 4)
(41, 5)


,video_id,template,placeholders,session,cam
0,261125-104457,C,PUCK,shellgame_mv_a,cam0
1,261125-104554,B,PUCK,shellgame_mv_a,cam0


In [4]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

index_df.template.value_counts()

template
C    18
A    16
B     7
Name: count, dtype: int64

In [5]:
index_df.session.value_counts()

session
shellgame_mv_a    18
shellgame_mv_c    12
shellgame_mv_b    11
Name: count, dtype: int64

### select videos

In [6]:
for i, row in index_df.iterrows():
    print(f"'{row['video_id']}', # --> {row['template']:5} -- {row['session']} ")


'261125-104457', # --> C     -- shellgame_mv_a 
'261125-104554', # --> B     -- shellgame_mv_a 
'261125-104705', # --> C     -- shellgame_mv_a 
'261125-104732', # --> A     -- shellgame_mv_a 
'261125-104759', # --> A     -- shellgame_mv_a 
'261125-104837', # --> C     -- shellgame_mv_a 
'261125-104917', # --> A     -- shellgame_mv_a 
'261125-104952', # --> A     -- shellgame_mv_a 
'261125-105943', # --> C     -- shellgame_mv_a 
'261125-110012', # --> C     -- shellgame_mv_a 
'261125-110038', # --> A     -- shellgame_mv_a 
'261125-110106', # --> A     -- shellgame_mv_a 
'261125-110135', # --> C     -- shellgame_mv_a 
'261125-110711', # --> A     -- shellgame_mv_a 
'261125-110736', # --> C     -- shellgame_mv_a 
'261125-110758', # --> C     -- shellgame_mv_a 
'261125-110830', # --> A     -- shellgame_mv_a 
'261125-110901', # --> B     -- shellgame_mv_a 
'261125-111934', # --> A     -- shellgame_mv_b 
'261125-112001', # --> C     -- shellgame_mv_b 
'261125-112026', # --> A     -- shellgam

In [7]:
vids_list = [
    '261125-104457', # --> C     -- shellgame_mv_a 
    '261125-104554', # --> B     -- shellgame_mv_a 
    '261125-104705', # --> C     -- shellgame_mv_a 
    '261125-104732', # --> A     -- shellgame_mv_a 
    '261125-104759', # --> A     -- shellgame_mv_a 
    '261125-104837', # --> C     -- shellgame_mv_a 
    '261125-104917', # --> A     -- shellgame_mv_a 
    '261125-104952', # --> A     -- shellgame_mv_a 
    '261125-105943', # --> C     -- shellgame_mv_a 
    '261125-110012', # --> C     -- shellgame_mv_a 
    '261125-110038', # --> A     -- shellgame_mv_a 
    '261125-110106', # --> A     -- shellgame_mv_a 
    '261125-110135', # --> C     -- shellgame_mv_a 
    '261125-110711', # --> A     -- shellgame_mv_a 
    '261125-110736', # --> C     -- shellgame_mv_a 
    '261125-110758', # --> C     -- shellgame_mv_a 
    '261125-110830', # --> A     -- shellgame_mv_a 
    '261125-110901', # --> B     -- shellgame_mv_a 
    '261125-111934', # --> A     -- shellgame_mv_b 
    '261125-112001', # --> C     -- shellgame_mv_b 
    '261125-112026', # --> A     -- shellgame_mv_b 
    '261125-112050', # --> B     -- shellgame_mv_b 
    '261125-112116', # --> C     -- shellgame_mv_b 
    '261125-112139', # --> A     -- shellgame_mv_b 
    '261125-112205', # --> B     -- shellgame_mv_b 
    '261125-112306', # --> C     -- shellgame_mv_b 
    '261125-112333', # --> A     -- shellgame_mv_b 
    '261125-112521', # --> C     -- shellgame_mv_b 
    '261125-112540', # --> C     -- shellgame_mv_b 
    '261125-114107', # --> C     -- shellgame_mv_c 
    '261125-114146', # --> A     -- shellgame_mv_c 
    '261125-114209', # --> C     -- shellgame_mv_c 
    '261125-114232', # --> B     -- shellgame_mv_c 
    '261125-114306', # --> C     -- shellgame_mv_c 
    '261125-114327', # --> B     -- shellgame_mv_c 
    '261125-114354', # --> C     -- shellgame_mv_c 
    '261125-114427', # --> A     -- shellgame_mv_c 
    '261125-114451', # --> A     -- shellgame_mv_c 
    '261125-114516', # --> C     -- shellgame_mv_c 
    '261125-114540', # --> B     -- shellgame_mv_c 
    '261125-114606', # --> A     -- shellgame_mv_c 
]
print(len(vids_list))

41


In [8]:
print(index_df.shape)
index_df = index_df[index_df.video_id.isin(vids_list)]
print(index_df.shape)

(41, 5)
(41, 5)


In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation 

### Compress Videos. 9second videos too long and high def. api call can handle <10MB vids and mine are well over 15mb per single view

#### 1. compress to lower res and store as mp4 (both cam0 and cam1)
#### 2. Concatenate the 2 views per video carefully - Because API cannot ingest 2 separate videos at the same time --> combined video for multiview evaluation

In [10]:
import os
import subprocess
from tqdm import tqdm

raw_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw"
compressed_dir = "data/compressed_videos"

os.makedirs(compressed_dir, exist_ok=True)

pbar = tqdm(vids_list, desc="Compressing Videos")
for vid in pbar:
    # Strip extension if vids_list already includes '.avi'
    stem = os.path.splitext(vid)[0]
    
    input_path = os.path.join(raw_dir, f"{stem}_cam1.avi")       # Change to f"{stem}_cam0.avi" if suffix is still used
    output_path = os.path.join(compressed_dir, f"{stem}_cam1.mp4")

    if os.path.exists(output_path):
        print(f"[-] Skipping (already exists): {stem}_cam1.mp4")
        continue

    if not os.path.exists(input_path):
        print(f"[!] Warning (missing input): {input_path}")
        continue

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", input_path,
            "-vf", "fps=20,scale=-2:480p",
            "-c:v", "libx264",
            "-crf", "23",
            "-an",
            output_path,
        ],
        check=True,
    )

    orig_mb = os.path.getsize(input_path) / (1024 * 1024)
    new_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[*] Converted {stem}: {orig_mb:.2f} MB -> {new_mb:.2f} MB")

Compressing Videos:   2%|▏         | 1/41 [01:07<45:12, 67.82s/it]

[*] Converted 261125-104457: 13.29 MB -> 1.81 MB


Compressing Videos:   5%|▍         | 2/41 [02:24<47:22, 72.90s/it]

[*] Converted 261125-104554: 14.66 MB -> 2.03 MB


Compressing Videos:   7%|▋         | 3/41 [03:31<44:36, 70.43s/it]

[*] Converted 261125-104705: 13.42 MB -> 1.90 MB


... (72 outputs left out)

Compressing Videos:  98%|█████████▊| 40/41 [34:04<01:34, 94.36s/it]

[*] Converted 261125-114540: 15.15 MB -> 2.21 MB


Compressing Videos: 100%|██████████| 41/41 [36:57<00:00, 54.08s/it] 

[*] Converted 261125-114606: 14.09 MB -> 2.17 MB


In [ ]:
## 31mins total to compress all 41 cam0 videos to 480p 20fps mp4 format.
## now doing same videos cam1 with a tqdm progressbar.

In [ ]:
import cv2

def concatenate_videos_side_by_side(video_path_0, video_path_1, output_path):
    """Stitches two synchronized videos horizontally into a single video file."""
    if os.path.exists(output_path):
        print(f"[-] Skipping concatenation: '{output_path}' already exists.")
        return
    
    cap0 = cv2.VideoCapture(video_path_0)
    cap1 = cv2.VideoCapture(video_path_1)

    fps = cap0.get(cv2.CAP_PROP_FPS) or 30.0
    width0 = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    height0 = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    height1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Match height of video 1 to video 0 if they differ
    target_height = height0

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    # Combined width = width of cam0 + width of cam1
    out_width = width0 + int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH) * (target_height / height1))
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_width, target_height))

    while cap0.isOpened() and cap1.isOpened():
        ret0, frame0 = cap0.read()
        ret1, frame1 = cap1.read()

        if not ret0 or not ret1:
            break

        # Resize frame1 if heights don't match exactly
        if frame1.shape[0] != target_height:
            new_w = int(frame1.shape[1] * (target_height / frame1.shape[0]))
            frame1 = cv2.resize(frame1, (new_w, target_height))

        # Concatenate horizontally (Left: View 0 | Right: View 1)
        combined_frame = cv2.hconcat([frame0, frame1])
        out.write(combined_frame)

    cap0.release()
    cap1.release()
    out.release()

    print(f"[*] New file created: {output_path}")

In [ ]:
videos_file_path = "data/compressed_videos/"
combined_videos_file_path = 'data/combined_videos/'

pbar = tqdm(vids_list, desc="Combining Videos")
for video_name in pbar:
    path_cam0 = f"{videos_file_path}{video_name}_cam0.mp4"
    path_cam1 = f"{videos_file_path}{video_name}_cam1.mp4"
    combined_path = f"{combined_videos_file_path}{video_name}_combined.mp4"

    # Stitch the videos horizontally
    concatenate_videos_side_by_side(path_cam0, path_cam1, combined_path)

In [ ]:
## combining videos loop ran in run.py

### Run one video

In [10]:
# Load environment variables from .env
load_dotenv()

True

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
)
client

In [12]:
# Encode combined video to base64
video_name = vids_list[0]

video_file_path = f'data/combined_videos/{video_name}_combined.mp4'

with open(video_file_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")
    b64_size_mb = len(b64_video) / (1024 ** 2)
    print(f"Base64 payload size: {b64_size_mb:.2f} MB")

Base64 payload size: 4.85 MB


In [13]:
len(b64_video)

5089376

In [14]:
assert not b64_video.startswith("data:"), "Duplicate data: prefix detected"
assert "\n" not in b64_video, "Newlines found in base64 string"

In [21]:
system_prompt = (
    "You are an expert multi-view visual object-tracking and spatial-temporal reasoning model. "
    "You are provided with a single synchronized split-screen video showing two side-by-side camera views of the exact same event: "
    "the LEFT half is 'cam0' and the RIGHT half is 'cam1'. "
    "In 'cam1', there are three clearly marked positions: A, B, and C, each starting with an inverted cup. "
    "At the start of the video, a puck is revealed under one of the cups in 'cam1'. "
    "The cups are then moved out of 'cam1' into 'cam0', shuffled inside 'cam0', and finally returned to 'cam1' onto markers A, B, and C. "
    "You must synthesize spatial and temporal cues across both camera views to continuously maintain the identity of the target cup across camera boundaries and determine its final resting position."
)

user_prompt = """
Analyze this synchronized split-screen video ('cam0' on the left, 'cam1' on the right) of a multi-view shell game and determine which position (A, B, or C) in 'cam1' hides the puck at the end of the video.

Valid Positions: {"A", "B", "C"}

Instructions:
1. Initial Reveal ('cam1' - Right): Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the beginning of the recording.
2. Track the cup with the puck chronologically:
  - Outbound Transition ('cam1' -> 'cam0'): Track the target cup as it leaves 'cam1' and enters 'cam0', noting its relative spatial order (e.g., left, middle, right) upon entering 'cam0'. 
  - Shuffle Tracking ('cam0' - Left): Continuously track the physical target cup through every swap and movement inside 'cam0'. 
  - Return Transition ('cam0' -> 'cam1'): Track the target cup as it exits 'cam0', re-enters 'cam1', and is placed onto one of the final markers (A, B, or C).
3. State your final predicted position ("A", "B", or "C") where the puck is located at the end of the video.
4. Provide a confidence score as a float between 0.0 and 1.0 denoting your certainty in the final predicted position (where 1.0 is completely certain and 0.33 is random guessing).
5. State the exact number of frames you sampled and processed from the given video in integer format.
6. Spatial Camera Setup Inference: Provide a single, concise sentence describing your best inference of the physical camera setup and the spatial position/orientation of 'cam0' relative to 'cam1' (e.g., adjacent adjacent top-down views, angled side-by-side cameras, or opposite-facing perspectives) based on how the cups and hands exit one frame and enter the other.

Output strictly in JSON matching this schema:
{
  "initial_position": "<A, B, or C>",
  "swap_sequence": "cam1:A -> enter_cam0:left -> cam0_swap:right -> return_cam1:B",
  "predicted_position": "<A, B, or C>",
  "confidence": 0.0,
  "frames_seen": 60,
  "camera_setup_reasoning": "<One sentence describing the inferred physical position and orientation of cam0 relative to cam1>"
}
"""
print(user_prompt)


Analyze this synchronized split-screen video ('cam0' on the left, 'cam1' on the right) of a multi-view shell game and determine which position (A, B, or C) in 'cam1' hides the puck at the end of the video.

Valid Positions: {"A", "B", "C"}

Instructions:
1. Initial Reveal ('cam1' - Right): Identify the starting position (A, B, or C) where the puck is initially revealed under a cup at the beginning of the recording.
2. Track the cup with the puck chronologically:
  - Outbound Transition ('cam1' -> 'cam0'): Track the target cup as it leaves 'cam1' and enters 'cam0', noting its relative spatial order (e.g., left, middle, right) upon entering 'cam0'. 
  - Shuffle Tracking ('cam0' - Left): Continuously track the physical target cup through every swap and movement inside 'cam0'. 
  - Return Transition ('cam0' -> 'cam1'): Track the target cup as it exits 'cam0', re-enters 'cam1', and is placed onto one of the final markers (A, B, or C).
3. State your final predicted position ("A", "B", or "C

In [22]:
prompt_english_words = len(system_prompt.split(' ')) + len(user_prompt.split(' '))
print(f"Words in both prompts: {prompt_english_words}")
print(f"Estimated Text Tokens: {prompt_english_words*1.333:.1f}")

Words in both prompts: 454
Estimated Text Tokens: 605.2


In [ ]:
start_time = time.time()

response = client.chat.completions.create(
    model="google/gemini-3.8-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": f"data:video/mp4;base64,{b64_video}",
                        "processing": "agentic",
                        },
                },
                {
                    "type": "text",
                    # "text": "Whats happening in this video? please describe scene and activity.",
                    "text": user_prompt,
                },

            ],
        },
    ],
    extra_body={
        "reasoning": {"effort": "medium"},
    },
    response_format={"type": "json_object"},
    temperature=0.1,
)

print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print(f"Cost: $ {response.usage.cost:.6f}")

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Prompt tokens: 45207
Completion tokens: 6553
Cost: $ 0.032427
Elapsed time: 72.25s


In [20]:
response.choices[0].message.content

'{\n  "initial_position": "A",\n  "swap_sequence": "cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C",\n  "predicted_position": "C",\n  "confidence": 0.98,\n  "frames_seen": 126,\n  "camera_setup_reasoning": "cam0 is positioned directly above cam1 along the table with the same top-down orientation, forming a continuous vertical extension of the surface where cups exit the top of cam1 and enter the bottom of cam0."\n}'

In [23]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json

{'initial_position': 'A',
 'swap_sequence': 'cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C',
 'predicted_position': 'C',
 'confidence': 0.98,
 'frames_seen': 126,
 'camera_setup_reasoning': 'cam0 is positioned directly above cam1 along the table with the same top-down orientation, forming a continuous vertical extension of the surface where cups exit the top of cam1 and enter the bottom of cam0.'}

In [19]:
response.usage

CompletionUsage(completion_tokens=6553, prompt_tokens=45207, total_tokens=51760, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=6085, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=38595, image_tokens=None, text_tokens=None, video_tokens=2040), cost=0.032427375, is_byok=False, cost_details={'upstream_inference_cost': 0.032427375, 'upstream_inference_prompt_cost': 0.007853625, 'upstream_inference_completions_cost': 0.02457375})

In [21]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names

## this try except is from my testing when i wasnt giving the user_prompt to the models and model was returning confidence like 'low','high'
# try: 
#     conf = float(prediction_json.get("confidence", 0.0))
# except ValueError:
#     conf = 0.0

run_result = {
    "video_file": os.path.basename(video_file_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "cost": usage.cost if usage else None,
    "initial_position": prediction_json.get("initial_position"),
    "predicted_position": prediction_json.get("predicted_position"),
    "confidence": float(prediction_json.get("confidence", 0.0)),
    # "confidence": conf,
    "swap_sequence": prediction_json.get("swap_sequence"),
    "frames_seen": prediction_json.get("frames_seen"),
    "camera_setup_reasoning": prediction_json.get("camera_setup_reasoning"),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/shellgame_predictions_testing_models.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261125-104457_combined.mp4",
  "model": "google/gemini-3.8-flash",
  "elapsed_seconds": 72.25,
  "prompt_tokens": 45207,
  "completion_tokens": 6553,
  "total_tokens": 51760,
  "cost": 0.032427375,
  "initial_position": "A",
  "predicted_position": "C",
  "confidence": 0.98,
  "swap_sequence": "cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C",
  "frames_seen": 126,
  "camera_setup_reasoning": "cam0 is positioned directly above cam1 along the table with the same top-down orientation, forming a continuous vertical extension of the surface where cups exit the top of cam1 and enter the bottom of cam0."
}


In [22]:
# 1. Load jsonl file directly into a DataFrame
df_testing_models = pd.read_json("data/vlm_eval_results/shellgame_predictions_testing_models.jsonl", lines=True)
pd.set_option('display.float_format', '{:,.3f}'.format)
print(df_testing_models.shape)
print(df_testing_models)
print(f"\nTotal Cost of Testing: $ {round(sum(df_testing_models.cost), 3)}")

(1, 13)
                   video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence                                                                      swap_sequence  frames_seen                                                                                                                                                                                           camera_setup_reasoning
0  261125-104457_combined.mp4  google/gemini-3.8-flash           72.250          45207               6553         51760 0.032                A                  C       0.980  cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C          126  cam0 is positioned directly above cam1 along the table with the same top-down orientation, forming a continuous vertical extension of the surface where cups exit the top of cam1 and enter the bottom of cam0.

Total Cost of Testing: $ 0.032


## Main Experiment: For loop over many videos

In [ ]:
## run_openrouter.py ranin tmux shell

# Loading and analyzing the jsonl outputs of the VLM
## Main Experiment: shellgame 
### gemini-3.8-flash (processing:'agentic')

In [9]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/shellgame_predictions_gemini-3.8-flash-agentic.jsonl", lines=True)

print(df.shape)
print(df.head(10))

(27, 13)
                   video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence                                                                                          swap_sequence  frames_seen                                                                                                                                                                                                                   camera_setup_reasoning
0  261125-104457_combined.mp4  google/gemini-3.8-flash            72.25          45207               6553         51760  0.03                A                  C        0.98                      cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C          126                          cam0 is positioned directly above cam1 along the table with the same top-down orientation, forming a continuous vertical extension of the surface where c

In [10]:
print(f'cost: $ {df.cost.sum():.3f}')

cost: $ 1.312


In [11]:
index_df['video_file'] = index_df.video_id + '_combined.mp4'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(41, 6)
(27, 18)
                    video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_position predicted_position  confidence                                                                      swap_sequence  frames_seen                                                                                                                                                                                           camera_setup_reasoning       video_id template placeholders         session   cam
11  261125-110736_combined.mp4  google/gemini-3.8-flash           100.00          21100              11699         32799  0.05                A                  C        0.98  cam1:A -> enter_cam0:left -> cam0_swap:middle -> cam0_swap:right -> return_cam1:C           79  cam0 is positioned further down the table facing the same direction as cam1, capturing the extended play area such that objects pushed out the top edge of cam1 enter direct

In [12]:
df['correct_top1'] = df['predicted_position'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 27
correct:   15
top1acc: 0.5555555555555556


In [13]:
import textwrap
import pandas as pd

pd.set_option('display.float_format', '{:,.3f}'.format)

def analyze_df(df):
    print('=' * 20, 'DF Stats by Session', '=' * 20)

    stats = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1', 'mean'),
        avg_confidence=('confidence', 'mean'),
        avg_prompt_tokens=('prompt_tokens', 'mean'),
        avg_completion_tokens=('completion_tokens', 'mean'),
        total_cost_usd=('cost', 'sum'),
        avg_frames_seen=('frames_seen', 'mean'),
        sd_frames_seen=('frames_seen', 'std')
    )
    print(stats)
    print('\n')

    print('=' * 30, 'Individual Videos by session', '=' * 30)

    for session in df['session'].unique():
        print('\n' + '=' * 30, session, '=' * 30)

        for i, row in df[df['session'] == session].iterrows():
            # Include ground-truth column if present in your merged DataFrame
            summary_cols = [
                col for col in [
                    'video_id', 'session', 'initial_position',
                    'template', 'predicted_position', 'confidence', 'correct_top1','frames_seen'
                ] if col in df.columns
            ]
            print(row[summary_cols].to_frame().T.to_string(index=False))

            print('\nSwap Sequence: ', textwrap.fill(str(row['swap_sequence']), width=100))
            print('\ncamera_setup_reasoning: ', textwrap.fill(str(row['camera_setup_reasoning']), width=100))
            print('-' * 60)

In [14]:
analyze_df(df)

==================== DF Stats by Session ====================
                videos  top1_acc  avg_confidence  avg_prompt_tokens  avg_completion_tokens  total_cost_usd  avg_frames_seen  sd_frames_seen
session                                                                                                                                    
shellgame_mv_a      16     0.750           0.983         37,977.062              9,452.938           0.698          115.562          40.405
shellgame_mv_b       8     0.375           0.980         48,687.750             12,063.375           0.453          117.625          23.421
shellgame_mv_c       3     0.000           0.980         29,941.333             12,718.667           0.162          101.000          66.731


============================== Individual Videos by session ==============================

============================== shellgame_mv_a ==============================
     video_id        session initial_position template predicted_po

In [15]:
print(len(set(vids_list) - set(df.video_id)))
set(vids_list) - set(df.video_id)

14


{'261125-104952',
 '261125-110758',
 '261125-112050',
 '261125-112333',
 '261125-112521',
 '261125-114107',
 '261125-114146',
 '261125-114209',
 '261125-114232',
 '261125-114354',
 '261125-114427',
 '261125-114451',
 '261125-114540',
 '261125-114606'}